In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arya
import pandas as pd

In [ ]:
import ana_utils
import filter_utils

In [ ]:
from astropy import units as u

In [ ]:
from dataclasses import dataclass

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
from astropy.table import Table

In [ ]:
import sys
sys.path.append("../photometry/")
from phot_utils import outer_join_xmatch
from calc_panstarrs_offset import get_panstarrs

In [ ]:
from copy import copy

In [ ]:
def get_default_params(objname):
    obs_props = ana_utils.get_obs_props(objname)
    return filter_utils.filter_params(
        dm = obs_props["dm"], 
        iso_fe_h = obs_props["iso_fe_h"],
        iso_log_age = np.log10(obs_props["iso_age"]),
        objname = objname, 
        mode="gri", 
        A_V = 0,
        r23_max_sigma=None,
        r_cen = 40/60,
        iso_width=0.15
    )
    

In [ ]:
ps_memb_kwargs = dict(
    s=4,
    lw=0.5, 
    color="w",
    ec="C3",
)

ps_all_kwargs = dict(
    s=2,
    lw=0.5, 
    color="w",
    ec="k",
)

In [ ]:
def plot_gr_background_fancy(subsets, params, results, depth_sigma=5):
    filter_utils.plot_gr_background(subsets, params)
    cat = subsets["all"]
    plot_gr_depth(cat, params, sigma=depth_sigma)
    
    df = subsets["background_all"]
    df = df[df["is_ps"]]
    plt.scatter(df["G_MAG_PS"] - df["R_MAG_PS"], df["G_MAG_PS"],  **ps_all_kwargs)

    df = subsets["background_selected"]
    df = df[df["is_ps"]]
    plt.scatter(df["G_MAG_PS"] - df["R_MAG_PS"], df["G_MAG_PS"],  **ps_memb_kwargs)

    plt.text(0.1, 0.9, f"${results.Nbkg_med:0.2f} \\pm {results.Nbkg_std:0.2f}$", transform=plt.gca().transAxes,
             color="C1", fontsize=8)
    plt.ylim(27, 16)


In [ ]:
def plot_gr_centre_fancy(subsets, params, results, depth_sigma=5):
    filter_utils.plot_gr_centre(subsets, params)
    df = subsets["centre_all"]
    df = df[df["is_ps"]]
    plt.scatter(df["G_MAG_PS"] - df["R_MAG_PS"], df["G_MAG_PS"],  **ps_all_kwargs)


    df = subsets["centre_selected"]
    df = df[df["is_ps"]]
    plt.scatter(df["G_MAG_PS"] - df["R_MAG_PS"], df["G_MAG_PS"],  **ps_memb_kwargs)

    plt.text(0.1, 0.9, f"${results.Ncen}$", transform=plt.gca().transAxes,
         color="C0", fontsize=8)
    cat = subsets["all"]

    plot_gr_depth(cat, params, sigma=depth_sigma)
    plt.ylim(27, 16)


In [ ]:
def plot_ri_background_fancy(subsets, params, depth_sigma=5):
    filter_utils.plot_ri_background(subsets, params)
    cat = subsets["all"]

    plot_ri_depth(cat, params, sigma=depth_sigma)
    
    df = subsets["background_all"]
    df = df[df["is_ps"]]
    plt.scatter(df["R_MAG"] - df["I_MAG"], df["R_MAG"],  **ps_all_kwargs)


    df = subsets["background_selected"]
    df = df[df["is_ps"]]
    plt.scatter(df["R_MAG"] - df["I_MAG"], df["R_MAG"],  **ps_memb_kwargs)

    plt.ylim(27, 16)


In [ ]:
def plot_ri_centre_fancy(subsets, params, depth_sigma=5):
    filter_utils.plot_ri_centre(subsets, params)
    df = subsets["centre_all"]
    df = df[df["is_ps"]]
    plt.scatter(df["R_MAG"] - df["I_MAG"], df["R_MAG"],  **ps_all_kwargs)

    df = subsets["centre_selected"]
    df = df[df["is_ps"]]
    plt.scatter(df["R_MAG"] - df["I_MAG"], df["R_MAG"],  **ps_memb_kwargs)
    cat = subsets["all"]

    plot_ri_depth(cat, params, sigma=depth_sigma)
    plt.ylim(27, 16)


In [ ]:
def detection_plot(cat, params, xi=1.5/60, eta=-1.5/60, depth_sigma=5):
    subsets = filter_utils.select_subsets(cat, params, xi=xi, eta=eta)
    results = filter_utils.catalog_stats(cat, params).iloc[0, :]

    fig, axs = plt.subplots(2, 3, figsize=(6, 4))

    plt.sca(axs[0][0])
    filter_utils.plot_tangent(cat, s=0.3, lw=0, color="k")
    plt.title("all")

    plt.sca(axs[1][0])
    filter_utils.plot_selected_points(subsets, params, xi=xi, eta=eta)
    filter_utils.plot_tangent(subsets["selected"][subsets["selected"]["is_ps"]], **ps_memb_kwargs)
    plt.gca().invert_xaxis()
    
    plt.sca(axs[1][1])
    plot_gr_background_fancy(subsets, params, results, depth_sigma=depth_sigma)


    plt.sca(axs[0][1])
    plot_gr_centre_fancy(subsets, params, results, depth_sigma=depth_sigma)

    plt.sca(axs[1][2])
    plot_ri_background_fancy(subsets, params, depth_sigma=depth_sigma)

    plt.sca(axs[0][2])
    plot_ri_centre_fancy(subsets, params, depth_sigma=depth_sigma)

    plt.tight_layout()


In [ ]:
def point_source_depth(mag_err_model, sigma):
    mag_err_0 = 2.5 / np.log(10) * 1/sigma

    return bisect(lambda x: mag_err_0 - mag_err_model(x), 10, 40)

In [ ]:
depth_sigma = 10

def get_g_depth(cat, params, sigma):
    g_err = ana_utils.fit_err(cat[params.gcol], cat[params.gcol + "_ERR"])
    g_depth = point_source_depth(g_err, sigma)
    return g_depth
    
def get_r_depth(cat, params, sigma):
    r_err = ana_utils.fit_err(cat[params.rcol], cat[params.rcol + "_ERR"])
    r_depth = point_source_depth(r_err, sigma)
    return r_depth

def get_i_depth(cat, params, sigma):
    i_err = ana_utils.fit_err(cat[params.icol], cat[params.icol + "_ERR"])
    i_depth = point_source_depth(i_err, sigma)
    return i_depth

In [ ]:
from scipy.optimize import bisect

In [ ]:
def plot_depth(r_depth, i_depth, sigma):
    x = np.linspace(-2, 2, 1000)
    mag_r = r_depth
    mag_i = np.minimum(i_depth, mag_r - x)
    mag_max = np.minimum(mag_r, mag_i +x)
    color=f"C0"
    plt.plot(x, mag_max, color=color)
    idx_0 = np.argmin(np.abs(x - -0.5))
    plt.annotate(xy=(x[idx_0], mag_max[idx_0]), text=f"${sigma} \\sigma$", color=color, fontsize=8,
                xytext=(0, 5), textcoords='offset points',)

def plot_ri_depth(cat, params, sigma):
    r_depth = get_r_depth(cat, params, sigma)
    i_depth = get_i_depth(cat, params, sigma)
    plot_depth(r_depth, i_depth, sigma)

def plot_gr_depth(cat, params, sigma):
    g_depth = get_g_depth(cat, params, sigma)
    r_depth = get_r_depth(cat, params, sigma)
    plot_depth(g_depth, r_depth, sigma)


In [ ]:
def correct_bright_stars(cat, obj, sat_g=18, sat_r=17.5, sat_i=18):

    cat_ps = get_panstarrs(obj)
    cat_ps = cat_ps[-cat_ps["iMeanKronMag"] + cat_ps["iMeanPSFMag"] < -0.02]
    cat_ps.rename_columns(
        ["G_MAG", "R_MAG", "I_MAG", "G_MAG_ERR", "R_MAG_ERR", "I_MAG_ERR", "ra", "dec", ],
        ["G_MAG_PS", "R_MAG_PS", "I_MAG_PS", "G_MAG_PS_ERR", "R_MAG_PS_ERR", "I_MAG_PS_ERR", "ra_ps", "dec_ps"]
    )


    cat = outer_join_xmatch(cat, cat_ps, ra1="ra", dec1="dec", ra2="ra_ps", dec2="dec_ps")
    
    filt = (cat["G_MAG"] < sat_g) | cat["G_MAG"].mask
    filt |= cat["R_MAG"] < sat_r | cat["G_MAG"].mask
    filt |= cat["I_MAG"] < sat_i | cat["G_MAG"].mask
    filt &= ~cat["G_MAG_PS"].mask
    filt &= ~cat["R_MAG_PS"].mask
    filt &= ~cat["I_MAG_PS"].mask

    if "A_g" in cat.colnames:
        cat["G_MAG_PS"] -= cat["A_g"]
        cat["R_MAG_PS"] -= cat["A_r"]
        cat["I_MAG_PS"] -= cat["A_i"]

    cat["G_MAG_OSIRIS"] = cat["G_MAG"]
    cat["R_MAG_OSIRIS"] = cat["R_MAG"]
    cat["I_MAG_OSIRIS"] = cat["I_MAG"]
    cat["G_MAG"][filt] = cat["G_MAG_PS"][filt]
    cat["R_MAG"][filt] = cat["R_MAG_PS"][filt]
    cat["I_MAG"][filt] = cat["I_MAG_PS"][filt]

    cat["R_FLAGS"][filt] = 0
    cat["R_FLAGS_WEIGHT"][filt] = 0
    cat["MAG_23"][filt] = -np.inf
    cat["is_ps"] = filt


    return cat

# Yasone 1

In [ ]:
params = get_default_params("yasone1")
params.A_V = 0.1
params.A_i_extra = 0.0
# params.r23_max_sigma = 3
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift", deredden=True)



In [ ]:
cat_psf = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift", deredden=True)


In [ ]:
filter_utils.calibrate_mag_col(cat, "MED_MAG_APER_LB_3")
filter_utils.calibrate_mag_col(cat, "MED_MAG_APER_3")
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")


In [ ]:
plt.scatter(cat_psf["G_MED_MAG_PSF"], cat_psf["G_MED_MAG_PSF_ERR"], s=1)
plt.ylim(0, 1)

In [ ]:
cat_ps = correct_bright_stars(cat, "yasone1", sat_r = 20)
for filt in ["G", "R", "I"]:
    cat_psf[f"{filt}_MAG"] = cat_psf[f"{filt}_MED_MAG_PSF"]
    cat_psf[f"{filt}_MAG_ERR"] = cat_psf[f"{filt}_MED_MAG_PSF_ERR"]
    
cat_ps_psf = correct_bright_stars(cat_psf, "yasone1", sat_r = 20)

In [ ]:
filter_utils.plot_with_hist(cat, params)

In [ ]:
detection_plot(cat_ps, params, xi=-2/60, eta=-2/60)

In [ ]:
detection_plot(cat_ps_psf, params, xi=-2/60, eta=-2/60)

In [ ]:
cat_ps_lb = cat_ps.copy()
cat_ps_lb["G_MAG"] = cat_ps_lb["G_MED_MAG_APER_LB_3"]
cat_ps_lb["R_MAG"] = cat_ps_lb["R_MED_MAG_APER_LB_3"]
cat_ps_lb["I_MAG"] = cat_ps_lb["I_MED_MAG_APER_LB_3"]


In [ ]:
detection_plot(cat_ps_lb, params, xi=-1/60, eta=-2/60)

In [ ]:
params.gcol = "G_MED_MAG_APER_3"
params.rcol = "R_MED_MAG_APER_3"
params.icol = "I_MED_MAG_APER_3"
detection_plot(cat_ps, params, xi=-1/60, eta=-2/60)

In [ ]:
cat_lss = Table.read("../survey_data/yasone1_lss10.csv")
xi, eta = ana_utils.to_tangent(ana_utils.to_coords(cat_lss), ana_utils.get_coord0("yasone1"))
cat_lss["xi"] = xi * 60 * u.arcmin
cat_lss["eta"] = eta * 60 * u.arcmin

cat_lss["mag_i"] = -100.0
cat_lss["mag_i_ERR"] = 0.05
cat_lss["mag_r_ERR"] = 2.5 / np.log(10) / cat_lss["snr_r"]
cat_lss["mag_g_ERR"] = 2.5 / np.log(10) / cat_lss["snr_g"]
cat_lss["R_SNR"] = cat_lss["snr_r"]
cat_lss["INDEX"] = np.arange(len(cat_lss))
cat_lss["is_ps"] = False
cat_lss["G_MAG_PS"] = np.nan
cat_lss["R_MAG_PS"] = np.nan
cat_lss["I_MAG_PS"] = np.nan
cat_lss = cat_lss[np.isfinite(cat_lss["mag_g"]) & np.isfinite(cat_lss["mag_r"])]

In [ ]:
params.gcol = "mag_g"
params.rcol = "mag_r"
params.icol = "mag_i"
params.flags_max = None
params.flags_weight_max = None
params.r23_max_sigma = None
params.detection_sigma = 5
params.mode = "gr"
params.A_V = 0.5
detection_plot(cat_lss, params, xi=-1/60, eta=-2/60)

# Yasone 2

In [ ]:
params = get_default_params("yasone2")
params.A_V = 0.00
params.r23_max_sigma = None

In [ ]:
from plot_utils import plot_mag_comparison

In [ ]:
cat = ana_utils.read_catalogue("yasone2", filter_bad=False, deredden=True, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")
cat_ps = correct_bright_stars(cat, "yasone2", sat_r = 20)

In [ ]:
cat_psf = ana_utils.read_catalogue("yasone2", filter_bad=False, deredden=True, catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift")


In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")
for filt in ["G", "R", "I"]:
    cat_psf[f"{filt}_MAG"] = cat_psf[f"{filt}_MED_MAG_PSF"]
    cat_psf[f"{filt}_MAG_ERR"] = cat_psf[f"{filt}_MED_MAG_PSF_ERR"]
    
cat_ps_psf = correct_bright_stars(cat_psf, "yasone2", sat_r = 20)

In [ ]:
filter_utils.calibrate_mag_col(cat, "MED_MAG_APER_LB_3")
filter_utils.calibrate_mag_col(cat, "MED_MAG_APER_3")


In [ ]:
cat["xi"] -= 0.0
cat["eta"] -= 0

In [ ]:
detection_plot(cat_ps, params, xi=3/60, eta=0.5/60)

In [ ]:
filter_utils.plot_with_hist(cat, params)

In [ ]:
detection_plot(cat_ps_psf, params, xi=3/60, eta=0.5/60)

In [ ]:
params.gcol = "G_MED_MAG_APER_LB_3"
params.rcol = "R_MED_MAG_APER_LB_3"
params.icol = "I_MED_MAG_APER_LB_3"
detection_plot(cat, params, xi=3/60, eta=0.5/60)

# Yasone 3

In [ ]:
params = get_default_params("yasone3")
params.iso_fe_h = "m2.00"
params.A_i_extra = -0.05
params.dm -= 0.5
params.A_V = 0.1

In [ ]:
cat = ana_utils.read_catalogue("yasone3", filter_bad=False, catname="allcolours", deredden=True)

In [ ]:
cat_ps = correct_bright_stars(cat, "yasone3", sat_r = 22)

In [ ]:
cat_psf = ana_utils.read_catalogue("yasone3", filter_bad=False, catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift", deredden=True)

In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")
for filt in ["G", "R", "I"]:
    cat_psf[f"{filt}_MAG"] = cat_psf[f"{filt}_MED_MAG_PSF"]
    cat_psf[f"{filt}_MAG_ERR"] = cat_psf[f"{filt}_MED_MAG_PSF_ERR"]
    
cat_ps_psf = correct_bright_stars(cat_psf, "yasone3", sat_r = 20)

In [ ]:
detection_plot(cat_ps, params, xi=1.0/60, eta=-2.5/60)

In [ ]:
detection_plot(cat_ps_psf, params, xi=1.0/60, eta=-2.5/60)

In [ ]:
filter_utils.plot_with_hist(cat, params, xi=-1/60, eta=2.5/60)

# Yasone 1 Extra

In [ ]:
cat_julen_sep = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="julen")

params_sep = copy(params)
params_sep.flags_max = None
params_sep.flags_weight_max = None
params_sep.r23_max_sigma = None
params_sep.A_V = 0.2
params_sep.A_i_extra += 0.1
params_sep.dm -= -0.1


In [ ]:
detection_plot(cat_julen_sep, params_sep)

In [ ]:
params_psf = copy(params)
params_psf.rcol = "R_MAG_PSF"
params_psf.gcol = "G_MAG_PSF"
params_psf.icol = "I_MAG_PSF"
params_psf.mode = "ri"


params_sep = copy(params)
params_sep.flags_max = None
params_sep.flags_weight_max = None
params_sep.r23_max_sigma = None


In [ ]:
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours")
cat_psf = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_psf")

cat_julen = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_julen_stack")
cat_julen_sep = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="julen")

In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MAG_PSF")

In [ ]:
for filt in ["G", "R", "I"]:
    cat_psf.rename_column(f"{filt}_MAGERR_PSF", f"{filt}_MAG_PSF_ERR")

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(4, 4), sharex=True, sharey=True)

plt.sca(axs[0][0])
subsets = filter_utils.select_subsets(cat, params, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_ri_centre(subsets, params)
plt.title("defaults")



plt.sca(axs[1][0])
subsets = filter_utils.select_subsets(cat_psf, params_psf, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_ri_centre(subsets, params_psf)
plt.title("psf mag")

plt.sca(axs[0][1])
subsets = filter_utils.select_subsets(cat_julen, params, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_ri_centre(subsets, params)

plt.title("julen image")


plt.sca(axs[1][1])
subsets = filter_utils.select_subsets(cat_julen_sep, params_sep, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_ri_centre(subsets, params_sep)

plt.title("julen + SEP")

plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(4, 4), sharex=True, sharey=True)

plt.sca(axs[0][0])
subsets = filter_utils.select_subsets(cat, params, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_gr_centre(subsets, params)
plt.title("defaults")



plt.sca(axs[1][0])
subsets = filter_utils.select_subsets(cat_psf, params_psf, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_gr_centre(subsets, params_psf)
plt.title("psf mag")

plt.sca(axs[0][1])
subsets = filter_utils.select_subsets(cat_julen, params, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_gr_centre(subsets, params)

plt.title("julen image")


plt.sca(axs[1][1])
subsets = filter_utils.select_subsets(cat_julen_sep, params_sep, xi=-1.5/60, eta=1.5/60)
filter_utils.plot_gr_centre(subsets, params_sep)

plt.title("julen + SEP")

plt.tight_layout()

In [ ]:
params = get_default_params("yasone1")
params.mode = "ri"
filter_utils.plot_with_hist(cat, params)

In [ ]:
params = get_default_params("yasone1")
filter_utils.plot_with_hist(cat, params)

In [ ]:
params

In [ ]:
params_new = copy(params)
params_new.A_V += 0.3
params_new.dm -= 0.3
params_new.mode = "ri"
filter_utils.plot_with_hist(cat, params_new)

In [ ]:
params_new.mode = "gri"
filter_utils.plot_with_hist(cat, params_new)